In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

import os

file_paths = []

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        file_paths.append(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ===== 0. Imports & cấu hình =====
# "Imports" = nạp các thư viện cần dùng (giống như include trong C/C++).
import os, random, time
from dataclasses import dataclass
from typing import Dict

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode

# ===== 0.1. Seed: giúp kết quả "ổn định" hơn khi chạy lại =====
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

# ===== 0.2. Chọn device =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device đang dùng:", device)

# ===== 1. Hyperparams =====
@dataclass
class CFG:
    batch_size: int = 64
    num_workers: int = 4
    head_lr: float = 5e-4
    backbone_lr: float = 1e-4
    epochs: int = 24
    weight_decay: float = 1e-4
    log_every: int = 50
    label_smoothing: float = 0.1


cfg = CFG()

if device.type == "cpu":
    cfg.num_workers = 0

print("Cấu hình đang dùng:", cfg)


In [ ]:
# ============================================================
# SESSION 1 — DATASET: 30 món ăn Việt Nam
#
# Mục tiêu:
# 1) Khai báo đường dẫn dataset trên Kaggle.
# 2) Kiểm tra cấu trúc thư mục Train / Validate / Test.
# 3) Đọc dataset bằng ImageFolder.
# 4) Lấy danh sách class món ăn và số lượng ảnh.
#
# Ghi chú:
# - Dataset phải có dạng:
#   Train/class_name/image.jpg
#   Validate/class_name/image.jpg
#   Test/class_name/image.jpg
# - ImageFolder tự gán nhãn theo tên thư mục class.
# ============================================================

import os
from pathlib import Path
from torchvision.datasets import ImageFolder

# ------------------------------------------------------------
# 1.1. Khai báo đường dẫn dataset
# ------------------------------------------------------------
DATA_DIR = Path("/kaggle/input/datasets/quandang/vietnamese-foods/Images")

TRAIN_DIR = DATA_DIR / "Train"
VAL_DIR   = DATA_DIR / "Validate"
TEST_DIR  = DATA_DIR / "Test"

print("DATA_DIR :", DATA_DIR)
print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR  :", VAL_DIR)
print("TEST_DIR :", TEST_DIR)


# ------------------------------------------------------------
# 1.2. Kiểm tra thư mục có tồn tại không
# ------------------------------------------------------------
for folder in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if folder.exists():
        print(f"OK: {folder}")
    else:
        print(f"ERROR: Không tìm thấy {folder}")


# ------------------------------------------------------------
# 1.3. Đọc dataset tạm bằng ImageFolder
# ------------------------------------------------------------
# Ở Session 2 ta sẽ thêm transform ảnh.
# Ở Session 1 chỉ đọc dataset để kiểm tra class và số lượng ảnh.
train_ds_raw = ImageFolder(TRAIN_DIR)
val_ds_raw   = ImageFolder(VAL_DIR)
test_ds_raw  = ImageFolder(TEST_DIR)


# ------------------------------------------------------------
# 1.4. Lấy thông tin class
# ------------------------------------------------------------
class_names = train_ds_raw.classes
class_to_idx = train_ds_raw.class_to_idx
num_classes = len(class_names)

print("\nSố class:", num_classes)
print("Class names:")
for i, name in enumerate(class_names):
    print(f"{i:02d}. {name}")

print("\nClass to index:")
print(class_to_idx)


# ------------------------------------------------------------
# 1.5. Kiểm tra số lượng ảnh
# ------------------------------------------------------------
print("\nSố lượng ảnh:")
print("Train:", len(train_ds_raw))
print("Val  :", len(val_ds_raw))
print("Test :", len(test_ds_raw))
print("Total:", len(train_ds_raw) + len(val_ds_raw) + len(test_ds_raw))


# ------------------------------------------------------------
# 1.6. Thống kê số ảnh mỗi class trong tập Train
# ------------------------------------------------------------
print("\nSố ảnh mỗi class trong Train:")

train_class_counts = {}

for class_name in class_names:
    class_folder = TRAIN_DIR / class_name

    image_files = [
        file for file in class_folder.iterdir()
        if file.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]
    ]

    train_class_counts[class_name] = len(image_files)
    print(f"{class_name:25s}: {len(image_files)} ảnh")


# ------------------------------------------------------------
# 1.7. Kiểm tra thử 1 ảnh đầu tiên
# ------------------------------------------------------------
sample_path, sample_label = train_ds_raw.samples[0]

print("\nSample đầu tiên:")
print("Path :", sample_path)
print("Label index:", sample_label)
print("Label name :", class_names[sample_label])

In [ ]:
# ============================================================
# SESSION 2 — TRANSFORM & DATALOADER
#
# Mục tiêu:
# 1) Chuẩn hóa ảnh đầu vào cho EfficientNet-B2.
# 2) Tạo transform cho Train / Validation / Test.
# 3) Tạo lại dataset có transform.
# 4) Tạo DataLoader để đưa dữ liệu vào model theo batch.
# ============================================================

import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# 2.1. Cấu hình cơ bản
# ------------------------------------------------------------
IMG_SIZE = 260
BATCH_SIZE = cfg.batch_size
NUM_WORKERS = cfg.num_workers

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


# ------------------------------------------------------------
# 2.2. Mean / Std theo ImageNet
# ------------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# ------------------------------------------------------------
# 2.3. Transform cho tập Train
# ------------------------------------------------------------
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.7, 1.0),
        interpolation=InterpolationMode.BICUBIC
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.25,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# ------------------------------------------------------------
# 2.4. Transform cho tập Validation / Test
# ------------------------------------------------------------
eval_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# ------------------------------------------------------------
# 2.5. Tạo Dataset có transform
# ------------------------------------------------------------
train_ds = ImageFolder(
    root=TRAIN_DIR,
    transform=train_transform
)

val_ds = ImageFolder(
    root=VAL_DIR,
    transform=eval_transform
)

test_ds = ImageFolder(
    root=TEST_DIR,
    transform=eval_transform
)

print("Dataset loaded successfully!")
print("Train:", len(train_ds))
print("Val  :", len(val_ds))
print("Test :", len(test_ds))


# ------------------------------------------------------------
# 2.6. Tạo DataLoader
# ------------------------------------------------------------
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    dataset=val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\nDataLoader created successfully!")
print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))


# ------------------------------------------------------------
# 2.7. Kiểm tra thử một batch
# ------------------------------------------------------------
images, labels = next(iter(train_loader))

print("\nOne batch:")
print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Image dtype :", images.dtype)
print("Label dtype :", labels.dtype)


In [ ]:
# ============================================================
# SESSION 3 — MODEL: EfficientNet-B2 Fine-Tuning
#
# Ý tưởng:
# - Vẫn dùng EfficientNet-B2 pretrained.
# - Không freeze toàn bộ backbone nữa.
# - Chỉ mở block cuối để model học đặc trưng riêng của món ăn Việt Nam.
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

# ------------------------------------------------------------
# 3.1. Load EfficientNet-B2 pretrained
# ------------------------------------------------------------
weights = models.EfficientNet_B2_Weights.IMAGENET1K_V1
tl_model = models.efficientnet_b2(weights=weights)

print("Loaded EfficientNet-B2 pretrained successfully!")


# ------------------------------------------------------------
# 3.2. Freeze toàn bộ model trước
# ------------------------------------------------------------
for param in tl_model.parameters():
    param.requires_grad = False


# ------------------------------------------------------------
# 3.3. Mở 2 block cuối để fine-tune
# ------------------------------------------------------------
for param in tl_model.features[-2:].parameters():
    param.requires_grad = True

print("\nUnfroze last 2 feature blocks for fine-tuning.")


# ------------------------------------------------------------
# 3.4. Thay classifier cho bài toán 30 class
# ------------------------------------------------------------
in_features = tl_model.classifier[1].in_features

tl_model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, num_classes)
)

print("\nClassifier mới:")
print(tl_model.classifier)


# ------------------------------------------------------------
# 3.5. Đưa model lên device
# ------------------------------------------------------------
tl_model = tl_model.to(device)

print("\nModel moved to:", device)


# ------------------------------------------------------------
# 3.6. Kiểm tra số lượng tham số trainable
# ------------------------------------------------------------
total_params = sum(p.numel() for p in tl_model.parameters())
trainable_params = sum(p.numel() for p in tl_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print("\nParameter summary:")
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Frozen params   : {frozen_params:,}")


# ------------------------------------------------------------
# 3.7. Test forward thử với một batch
# ------------------------------------------------------------
images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    logits = tl_model(images)

print("\nForward test:")
print("Input images shape:", images.shape)
print("Output logits shape:", logits.shape)


In [ ]:
# ============================================================
# SESSION 4 — TRAINING EfficientNet-B2 cho món ăn Việt Nam
# ============================================================

import copy
import time
import torch
import torch.nn as nn

amp_device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# 4.1. Cấu hình training
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)

head_lr = cfg.head_lr
backbone_lr = cfg.backbone_lr
weight_decay = cfg.weight_decay
epochs = cfg.epochs
log_every = cfg.log_every

save_path = "/kaggle/working/best_efficientnet_b2_30vnfoods_finetuned.pth"

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler(amp_device, enabled=use_amp)

print("Training config:")
print("Head LR      :", head_lr)
print("Backbone LR  :", backbone_lr)
print("Weight decay :", weight_decay)
print("Epochs       :", epochs)
print("Label smooth :", cfg.label_smoothing)
print("Use AMP      :", use_amp)
print("Save path    :", save_path)


# ------------------------------------------------------------
# 4.2. Hàm đếm số dự đoán đúng
# ------------------------------------------------------------
@torch.no_grad()
def count_correct(logits, y):
    pred = logits.argmax(dim=1)
    return (pred == y).sum().item()


# ------------------------------------------------------------
# 4.3. Train 1 epoch
# ------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, criterion, epoch):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    t0 = time.time()

    for step, (x, y) in enumerate(loader, start=1):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        batch_size = y.size(0)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=amp_device, enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * batch_size
        total_correct += count_correct(logits, y)
        total_samples += batch_size

        if step % log_every == 0:
            avg_loss = total_loss / total_samples
            avg_acc = total_correct / total_samples

            print(
                f"Epoch {epoch:02d} | "
                f"step {step:4d}/{len(loader)} | "
                f"loss {avg_loss:.4f} | "
                f"acc {avg_acc:.4f} | "
                f"time {time.time() - t0:.1f}s"
            )

    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples
    return epoch_loss, epoch_acc


# ------------------------------------------------------------
# 4.4. Evaluate model
# ------------------------------------------------------------
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        batch_size = y.size(0)

        with torch.amp.autocast(device_type=amp_device, enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)

        total_loss += loss.item() * batch_size
        total_correct += count_correct(logits, y)
        total_samples += batch_size

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc


# ------------------------------------------------------------
# 4.5. Hàm train chính
# ------------------------------------------------------------
def train_model(model, model_name="EfficientNet-B3"):
    model = model.to(device)

    print("\n" + "=" * 70)
    print(f"TRAINING: {model_name}")
    print("=" * 70)

    optimizer = torch.optim.AdamW(
        [
            {"params": tl_model.features[-2:].parameters(), "lr": backbone_lr},
            {"params": tl_model.classifier.parameters(), "lr": head_lr},
        ],
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs
    )

    best_val_acc = 0.0
    best_epoch = 0
    best_state = copy.deepcopy(model.state_dict())

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        print("-" * 70)

        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            epoch=epoch
        )

        val_loss, val_acc = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion
        )

        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        head_current_lr = optimizer.param_groups[1]["lr"]
        backbone_current_lr = optimizer.param_groups[0]["lr"]

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

            torch.save({
                "epoch": best_epoch,
                "model_name": model_name,
                "model_state_dict": best_state,
                "class_names": class_names,
                "class_to_idx": class_to_idx,
                "num_classes": num_classes,
                "val_acc": best_val_acc,
                "history": history,
                "img_size": IMG_SIZE,
                "head_lr": head_lr,
                "backbone_lr": backbone_lr,
                "weight_decay": weight_decay
            }, save_path)

            print("Saved best model!")

        print(
            f"Epoch {epoch:02d} summary | "
            f"train loss {train_loss:.4f} | "
            f"train acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} | "
            f"val acc {val_acc:.4f} | "
            f"best val acc {best_val_acc:.4f} | "
            f"backbone lr {backbone_current_lr:.6f} | "
            f"head lr {head_current_lr:.6f}"
        )

    model.load_state_dict(best_state)

    print("\nTraining completed!")
    print(f"Best epoch   : {best_epoch}")
    print(f"Best val acc : {best_val_acc:.4f}")
    print(f"Best model saved at: {save_path}")

    return model, best_val_acc, history


# ------------------------------------------------------------
# 4.6. Chạy training
# ------------------------------------------------------------
tl_model, best_val_acc, history = train_model(
    model=tl_model,
    model_name="EfficientNet-B2 Fine-Tuned"
)


# ------------------------------------------------------------
# 4.7. Đánh giá trên test set
# ------------------------------------------------------------
test_loss, test_acc = evaluate(
    model=tl_model,
    loader=test_loader,
    criterion=criterion
)

print("\nFINAL RESULT")
print("=" * 70)
print(f"Best Val Acc : {best_val_acc:.4f}")
print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Acc     : {test_acc:.4f}")


In [ ]:
# ============================================================
# SESSION 5 — EVALUATION & VISUALIZATION (Inference)
#
# Mục tiêu:
# 1) Không chỉ nhìn accuracy, mà xem thêm precision / recall / f1 theo từng class.
# 2) Xem confusion matrix để biết các món nào dễ bị nhầm với nhau.
# 3) Tính thêm top-3 accuracy vì nhiều món ăn khá giống nhau.
# 4) Hiển thị ảnh dự đoán đúng / sai để hiểu model hoạt động ra sao.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F

from sklearn.metrics import classification_report, confusion_matrix


# ------------------------------------------------------------
# 5.0. Mean / Std để unnormalize ảnh khi visualize
# ------------------------------------------------------------
IMAGENET_MEAN_TENSOR = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
IMAGENET_STD_TENSOR = torch.tensor(IMAGENET_STD).view(3, 1, 1)


def unnormalize(img_tensor):
    return img_tensor * IMAGENET_STD_TENSOR + IMAGENET_MEAN_TENSOR


# ------------------------------------------------------------
# 5.1. Predict 1 batch
# ------------------------------------------------------------
@torch.no_grad()
def predict_batch(model, x):
    model.eval()
    logits = model(x)
    probs = F.softmax(logits, dim=1)
    pred_top1 = probs.argmax(dim=1)
    top3_probs, top3_idx = probs.topk(k=min(3, probs.size(1)), dim=1)
    return logits, probs, pred_top1, top3_idx, top3_probs


# ------------------------------------------------------------
# 5.2. Chạy inference trên toàn bộ loader
# ------------------------------------------------------------
@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()

    all_logits = []
    all_probs = []
    all_preds = []
    all_targets = []
    all_top3_idx = []
    all_top3_probs = []
    all_images = []
    all_paths = []

    start_idx = 0

    for x, y in loader:
        batch_size = y.size(0)

        x_device = x.to(device, non_blocking=True)
        logits, probs, preds, top3_idx, top3_probs = predict_batch(model, x_device)

        all_logits.append(logits.cpu())
        all_probs.append(probs.cpu())
        all_preds.append(preds.cpu())
        all_targets.append(y.cpu())
        all_top3_idx.append(top3_idx.cpu())
        all_top3_probs.append(top3_probs.cpu())
        all_images.append(x.cpu())

        if hasattr(loader.dataset, "samples"):
            batch_paths = [loader.dataset.samples[i][0] for i in range(start_idx, start_idx + batch_size)]
        else:
            batch_paths = [f"sample_{i}" for i in range(start_idx, start_idx + batch_size)]
        all_paths.extend(batch_paths)
        start_idx += batch_size

    return {
        "logits": torch.cat(all_logits),
        "probs": torch.cat(all_probs),
        "preds": torch.cat(all_preds),
        "targets": torch.cat(all_targets),
        "top3_idx": torch.cat(all_top3_idx),
        "top3_probs": torch.cat(all_top3_probs),
        "images": torch.cat(all_images),
        "paths": all_paths,
    }


# ------------------------------------------------------------
# 5.3. Tính metric tổng quan
# ------------------------------------------------------------
def compute_summary_metrics(results):
    y_true = results["targets"].numpy()
    y_pred = results["preds"].numpy()
    top3_idx = results["top3_idx"].numpy()

    top1_acc = (y_true == y_pred).mean()
    top3_acc = np.mean([y_true[i] in top3_idx[i] for i in range(len(y_true))])

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(y_true, y_pred)

    return {
        "top1_acc": top1_acc,
        "top3_acc": top3_acc,
        "report": report,
        "cm": cm,
    }


def print_summary(metrics):
    print("\n" + "=" * 80)
    print("SESSION 5 — TEST EVALUATION SUMMARY")
    print("=" * 80)
    print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f}")
    print(f"Top-3 Accuracy : {metrics['top3_acc']:.4f}")
    print(f"Macro Precision: {metrics['report']['macro avg']['precision']:.4f}")
    print(f"Macro Recall   : {metrics['report']['macro avg']['recall']:.4f}")
    print(f"Macro F1-score : {metrics['report']['macro avg']['f1-score']:.4f}")
    print(f"Weighted F1    : {metrics['report']['weighted avg']['f1-score']:.4f}")


# ------------------------------------------------------------
# 5.4. Bảng class mạnh / yếu nhất
# ------------------------------------------------------------
def class_report_dataframe(metrics):
    df = pd.DataFrame(metrics["report"]).T
    df = df.loc[class_names, ["precision", "recall", "f1-score", "support"]]
    df = df.sort_values("f1-score", ascending=False)
    return df


def show_best_worst_classes(df, top_k=10):
    print("\nTop classes theo F1-score:")
    display(df.head(top_k))

    print("\nWorst classes theo F1-score:")
    display(df.tail(top_k).sort_values("f1-score"))


# ------------------------------------------------------------
# 5.5. Vẽ confusion matrix
# ------------------------------------------------------------
def plot_confusion_matrix(cm, normalize=False, figsize=(18, 15)):
    if normalize:
        cm_plot = cm.astype(np.float64) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
        fmt = ".2f"
        title = "Normalized Confusion Matrix"
    else:
        cm_plot = cm
        fmt = "d"
        title = "Confusion Matrix (Counts)"

    plt.figure(figsize=figsize)
    sns.heatmap(
        cm_plot,
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        annot=False,
        fmt=fmt,
    )
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 5.6. Tạo bảng các mẫu dự đoán
# ------------------------------------------------------------
def build_prediction_dataframe(results):
    rows = []

    for i in range(len(results["targets"])):
        true_idx = int(results["targets"][i].item())
        pred_idx = int(results["preds"][i].item())
        top3_ids = results["top3_idx"][i].tolist()
        top3_ps = results["top3_probs"][i].tolist()

        rows.append({
            "index": i,
            "path": results["paths"][i],
            "true_idx": true_idx,
            "true_name": class_names[true_idx],
            "pred_idx": pred_idx,
            "pred_name": class_names[pred_idx],
            "correct": true_idx == pred_idx,
            "top1_prob": float(results["probs"][i, pred_idx].item()),
            "true_prob": float(results["probs"][i, true_idx].item()),
            "top3_names": [class_names[j] for j in top3_ids],
            "top3_probs": [float(p) for p in top3_ps],
        })

    return pd.DataFrame(rows)


def show_most_confident_errors(pred_df, top_k=15):
    wrong_df = pred_df[pred_df["correct"] == False].copy()
    wrong_df = wrong_df.sort_values("top1_prob", ascending=False)

    print("\nNhững lỗi model sai nhưng rất tự tin:")
    display(wrong_df.head(top_k)[[
        "path", "true_name", "pred_name", "top1_prob", "true_prob", "top3_names"
    ]])


# ------------------------------------------------------------
# 5.7. Hiển thị ảnh dự đoán đúng / sai
# ------------------------------------------------------------
def show_predictions(results, pred_df, max_images=12, only_wrong=False, sort_by_confidence=False):
    if only_wrong:
        df = pred_df[pred_df["correct"] == False].copy()
        if len(df) == 0:
            print("Không có ảnh dự đoán sai.")
            return
    else:
        df = pred_df.copy()

    if sort_by_confidence:
        df = df.sort_values("top1_prob", ascending=False)

    df = df.head(max_images)

    cols = 4
    rows = int(np.ceil(len(df) / cols))
    plt.figure(figsize=(4.5 * cols, 4.8 * rows))

    for plot_idx, row in enumerate(df.itertuples(index=False), start=1):
        img = unnormalize(results["images"][row.index]).clamp(0, 1)
        img = img.permute(1, 2, 0).numpy()

        top3_text = "\n".join(
            [f"{name}: {prob:.2f}" for name, prob in zip(row.top3_names, row.top3_probs)]
        )

        title = (
            f"True: {row.true_name}\n"
            f"Pred: {row.pred_name} ({row.top1_prob:.2f})\n"
            f"Top-3:\n{top3_text}"
        )

        plt.subplot(rows, cols, plot_idx)
        plt.imshow(img)
        plt.title(title, fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 5.8. Chạy toàn bộ evaluation trên test set
# ------------------------------------------------------------
test_results = collect_predictions(tl_model, test_loader)
test_metrics = compute_summary_metrics(test_results)
test_report_df = class_report_dataframe(test_metrics)
test_pred_df = build_prediction_dataframe(test_results)

print_summary(test_metrics)
show_best_worst_classes(test_report_df, top_k=10)
show_most_confident_errors(test_pred_df, top_k=15)

print("\nConfusion matrix dạng số đếm:")
plot_confusion_matrix(test_metrics["cm"], normalize=False, figsize=(18, 15))

print("\nConfusion matrix dạng chuẩn hóa theo từng class thật:")
plot_confusion_matrix(test_metrics["cm"], normalize=True, figsize=(18, 15))

print("\nMột số ảnh dự đoán đúng / tự tin:")
show_predictions(test_results, test_pred_df, max_images=12, only_wrong=False, sort_by_confidence=True)

print("\nMột số ảnh dự đoán sai / model tự tin:")
show_predictions(test_results, test_pred_df, max_images=12, only_wrong=True, sort_by_confidence=True)
